# ICT Daily Quiz 2028 — Restore missing Google Drive crops

This notebook crops the **original PDF pairs** using the same Python code as the live site. It validates **95 quizzes / 950 WEBP images** and uploads only the missing copies to the existing `Quiz XX / Questions` and `Quiz XX / Markings` Google Drive folders.

**Instructions:** In Google Colab, select **Runtime → Run all**, then sign in to the Google account that has edit access to the quiz Drive folders. Do not paste an API key or service-account secret. Re-run safely if interrupted; existing nonempty files are preserved. This workflow does not depend on GitHub Actions.

In [ ]:
from pathlib import Path
import os, subprocess, sys
repo = Path('/content/ict-daily-quiz')
url = 'https://github.com/adnimeshakalhara04-create/ict-daily-quiz.git'
if not (repo / '.git').exists():
    subprocess.run(['git','clone','--depth','1',url,str(repo)],check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pymupdf','pillow','google-api-python-client','google-auth'],check=True)
os.chdir(repo)
print('Original source PDFs and cropping code ready.')

In [ ]:
from google.colab import auth
auth.authenticate_user()
import google.auth
from googleapiclient.discovery import build
credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
service = build('drive','v3',credentials=credentials,cache_discovery=False)
for folder_id in ('1I9fU_ojRQBS-U3_5slnkWGN_SIsgRjmN','1mYROVvho24eDc3ys_OA2C2dmDq5cZ9NL'):
    meta = service.files().get(fileId=folder_id, fields='name,mimeType').execute()
    assert meta['mimeType']=='application/vnd.google-apps.folder'
    print('Drive access confirmed:',meta['name'])

In [ ]:
from PIL import Image
import json
repair = repo / 'repairs'
for old in repair.glob('q14-marking-q02.part*.b64'):
    old.unlink()
parts = sorted((repair / 'q14-clean').glob('part*'))
assert parts, 'Missing Quiz 14 image repair parts'
(repair / 'q14-marking-q02.part0.b64').write_bytes(b''.join(p.read_bytes() for p in parts))
subprocess.run([sys.executable,'scripts/process_incoming_auto.py'],cwd=repo,check=True)
data = json.loads((repo/'quiz-data.json').read_text(encoding='utf-8'))
assert (data['quizCount'],data['questionCount'])==(95,475)
count=0
for n in range(1,96):
    for kind in ('questions','markings'):
        for q in range(1,6):
            path=repo/'daily_assets'/kind/f'quiz-{n:02d}'/f'q-{q:02d}.webp'
            assert path.stat().st_size>1000,path
            with Image.open(path) as im: im.verify()
            count+=1
assert count==950
print('All 950 WEBP files cropped from the source PDFs and verified.')

In [ ]:
os.environ['REPAIR_FROM']='27'
os.environ['REPAIR_TO']='95'
subprocess.run([sys.executable,'scripts/repair_drive_existing.py'],cwd=repo,check=True)
print('Drive repairs uploaded and verified. Existing valid files remain untouched.')

In [ ]:
from urllib.request import urlopen
with urlopen('https://ict-daily-quiz-2028-nimesha.vercel.app/quiz-data.json',timeout=30) as r:
    public_data=json.load(r)
assert (public_data['quizCount'],public_data['questionCount'])==(95,475)
print('Production website verified: 95 quizzes / 475 questions.')